# Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime
import itertools

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from scipy import stats
from scipy.integrate import trapezoid
from scipy.stats import gaussian_kde, mannwhitneyu, ks_2samp, pearsonr, spearmanr, kruskal
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, confusion_matrix

from utils import *

warnings.filterwarnings('ignore')

# Funções auxiliares

Todas as funções do estudo ficam em **`utils.py`**, importado na célula acima com `from utils import *`.

| Bloco | Funções |
|---|---|
| Filtro de Hampel | `hampel`, `ResultadoHampel` |
| Carga e limpeza | `carregar_crystallizer`, `aplicar_tratamentos`, `carregar_eventos` |
| Janelas e eventos | `valores_na_janela`, `adicionar_eventos_ultrapassagem`, `separar_eventos_por_data`, `unificar_eventos` |
| Inspeção (fonte dos eventos) | `carregar_inspecoes`, `identificar_lacunas`, `identificar_paradas_de_planta`, `montar_tabela_eventos`, `eventos_para_notebook` |
| Limpeza de excursões | `classificar_amostras_altas`, `limpar_excursoes` |
| Baseline e detectores | `avaliar_baseline_lift`, `criterios_padrao`, `avaliar_detector_diario`, `avaliar_detector_composto`, `avaliar_detector_adaptativo`, `comparar_detectores` |
| Gráficos da série | `plot_crystallizer`, `plot_crystallizers`, `plot_mm_crystallizer`, `plot_mm_crystallizers`, `plot_crystallizer_unificado` |
| Estatística | `estatisticas`, `plot_violin_classes`, `plot_violin_serie_vs_ultrapassagem`, `testar_classes_por_janela`, `separabilidade_features` |
| Não supervisionado | `gerar_janelas_deslizantes`, `janelas_nas_falhas`, `clusterizar_regimes`, `plotar_regimes_pca`, `calcular_margem_cross_reator`, `baseline_movel`, `avaliar_lift_series`, `campanhas_por_reator`, `perfil_hazard_campanha` |
| Classificação | `extrair_features`, `dividir_dados`, `selecionar_features`, `definir_modelos_e_grids`, `avaliar_cv_com_grid`, `avaliar_teste`, `rodar_classificacao` |
| Detecção de anomalia | `rodar_iforest_janelas`, `plotar_iforest_janelas` |
| Cartas de controle | `calcular_ewma`, `otimizar_ewma`, `plotar_carta_ewma`, `avaliar_carta_controle`, `calcular_cusum_dinamico`, `otimizar_cusum_dinamico`, `plotar_carta_cusum` |

# Carregando dados

## Política de tratamento de dados — onde cada versão pode ser usada

O sinal que antecipa falha é justamente a leitura alta de ferro: **o outlier é o alvo**, e filtro
estatístico de outlier e detector de falha são a mesma operação com sinais trocados. Verificação
feita contra as duas leituras que a planilha de inspeção confirma como **causa** de parada de
emergência (C3 264 ppm em 23/11/2013 e C3 999 ppm em 02/08/2018, ambas com furo confirmado na
abertura) e contra o detector diário sobre as 31 falhas ancoradas:

| Tratamento | Leituras-gatilho | Máximo do C3 | Amostras >10 ppm (C3) | Detector max>10 (F2) | Detector mediana≥3.5 (F2) |
|---|---|---|---|---|---|
| Original + `limpar_excursoes` | **2/2 mantidas** | 999 ppm | 30 | **0.200** | 0.230 |
| Intervalo 0-10 | 0/2 | 10.0 ppm | 0 | 0.000 | 0.233 |
| IQR | 0/2 | 4.0 ppm | 0 | 0.000 | 0.236 |
| Hampel 15d | 0/2 | 6.9 ppm | 0 | 0.000 | 0.194 |
| Hampel 90d | 0/2 | 4.7 ppm | 0 | 0.000 | 0.154 |

Duas conclusões:

1. **Todo filtro estatístico deleta o alvo.** No IQR o maior valor sobrevivente é 4.0 ppm —
   abaixo do próprio limite operacional de 5 ppm: avaliar a regra vigente sobre dados IQR é
   impossível por construção.
2. **A estatística robusta substitui o filtro.** A mediana diária entrega o mesmo F2 sobre dados
   brutos e filtrados — o filtro não melhora o detector robusto e destrói o detector de cauda.

**Decisão (duas pistas):**

- **Pista A — canônica** (tabelas de evento, features, modelos, detector, baseline): sempre
  `Original` = bruto + `limpar_excursoes`, e `limpar_excursoes` descarta **apenas o valor
  extremo** (acima de 1000 ppm — uma única amostra em toda a base, 20000 ppm no C2). A regra
  causal (corroboração na vizinhança OU parada de amostragem em seguida) continua sendo
  calculada, mas como **rótulo de diagnóstico**, não como filtro: a leitura alta isolada fica
  na série porque é uma ultrapassagem sem falha, ou seja, o **falso positivo** que o modelo
  precisa aprender a rejeitar. Robustez a spike espúrio vem de features robustas (mediana,
  p75/p90) e das features relativas — não de pré-filtro.
- **Pista B — visualização e EDA distribucional**: `Intervalo 0-10`, `IQR` e `Hampel` continuam
  existindo para gráficos de série, KDE/violin e comparação descritiva (sem eles, 999 ppm
  esmaga qualquer escala). **Nunca alimentam janela de evento nem modelo.**

As células que comparam tratamentos lado a lado (estatística descritiva, effect size por
tratamento, violins por classe) foram mantidas porque são a evidência dessa decisão.

In [ ]:
# Parâmetros do filtro de Hampel (janela em amostras = dias * amostras por dia)
dias             = 15   # janela curta
dias_longo       = 90   # janela longa, usada para limpar a série completa
amostras_por_dia = 5    # aproximação: a base tem ~6 amostras de laboratório por dia

window_size = dias * amostras_por_dia

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

# Linhas 23 e 2141 estão duplicadas mas não possuem amostras significativas (i.e amostras com valores muito baixos)
df_crystallizer1, df_duplicados_crystallizer1 = carregar_crystallizer(
    base_name_crystallizer1, linhas_remover=[23, 2141], limite_ppm=None
)

# limite_ppm=None desliga o corte fixo. `limpar_excursoes` descarta APENAS o valor EXTREMO
# (acima do teto de implausibilidade de 1000 ppm). As leituras altas isoladas continuam na
# série mesmo quando a regra causal as classifica como provável erro de laboratório: uma
# ultrapassagem alta que NÃO terminou em falha é justamente o falso positivo que o modelo
# precisa aprender a rejeitar — e é o mais difícil deles. Ver a seção "Erro de laboratório
# vs medição alta informativa" para o veredito amostra a amostra.
df_crystallizer1, df_descartes_crystallizer1 = limpar_excursoes(
    df_crystallizer1, descartar_isoladas=False
)

tratamentos_crystallizer1, info_crystallizer1 = aplicar_tratamentos(
    df_crystallizer1,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer1

### Removendo outliers 0 a 10 #1

In [ ]:
df_crystallizer1_0a10 = tratamentos_crystallizer1["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer1)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer1_0a10)} ({len(df_crystallizer1) - len(df_crystallizer1_0a10)} removidas)")

df_crystallizer1_0a10

### Removendo outliers com IQR #1

In [ ]:
df_crystallizer1_iqr = tratamentos_crystallizer1["IQR"]
limite_inf, limite_sup = info_crystallizer1["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer1)}")
print(f"Amostras removidas: {len(df_crystallizer1) - len(df_crystallizer1_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer1_iqr)}")

df_crystallizer1_iqr

### Removendo outliers com Filtro de Hampel #1

In [ ]:
df_crystallizer1_hampel = tratamentos_crystallizer1["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer1['outliers_hampel']}")
df_crystallizer1_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #1

In [ ]:
df_crystallizer1_hampel90d = tratamentos_crystallizer1["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer1['outliers_hampel_longo']}")
df_crystallizer1_hampel90d

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

# A amostra de 20000 ppm (Labref 4027521) é o único descarte da base inteira — cai no teto
# de implausibilidade. A segunda maior leitura de toda a base é 999 ppm, confirmada por furo.
df_crystallizer2, df_duplicados_crystallizer2 = carregar_crystallizer(
    base_name_crystallizer2, limite_ppm=None
)

# limite_ppm=None desliga o corte fixo. `limpar_excursoes` descarta APENAS o valor EXTREMO
# (acima do teto de implausibilidade de 1000 ppm). As leituras altas isoladas continuam na
# série mesmo quando a regra causal as classifica como provável erro de laboratório: uma
# ultrapassagem alta que NÃO terminou em falha é justamente o falso positivo que o modelo
# precisa aprender a rejeitar — e é o mais difícil deles. Ver a seção "Erro de laboratório
# vs medição alta informativa" para o veredito amostra a amostra.
df_crystallizer2, df_descartes_crystallizer2 = limpar_excursoes(
    df_crystallizer2, descartar_isoladas=False
)

tratamentos_crystallizer2, info_crystallizer2 = aplicar_tratamentos(
    df_crystallizer2,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer2

### Removendo outliers 0 a 10 #2

In [ ]:
df_crystallizer2_0a10 = tratamentos_crystallizer2["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer2)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer2_0a10)} ({len(df_crystallizer2) - len(df_crystallizer2_0a10)} removidas)")

df_crystallizer2_0a10

### Removendo outliers com IQR #2

In [ ]:
df_crystallizer2_iqr = tratamentos_crystallizer2["IQR"]
limite_inf, limite_sup = info_crystallizer2["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer2)}")
print(f"Amostras removidas: {len(df_crystallizer2) - len(df_crystallizer2_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer2_iqr)}")

df_crystallizer2_iqr

### Removendo outliers com Filtro de Hampel #2

In [ ]:
df_crystallizer2_hampel = tratamentos_crystallizer2["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer2['outliers_hampel']}")
df_crystallizer2_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #2

In [ ]:
df_crystallizer2_hampel90d = tratamentos_crystallizer2["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer2['outliers_hampel_longo']}")
df_crystallizer2_hampel90d

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

# Linhas 6476 e 6494 estão duplicadas mas não possuem amostras significativas
df_crystallizer3, df_duplicados_crystallizer3 = carregar_crystallizer(
    base_name_crystallizer3, linhas_remover=[6476, 6494], limite_ppm=None
)

# limite_ppm=None desliga o corte fixo. `limpar_excursoes` descarta APENAS o valor EXTREMO
# (acima do teto de implausibilidade de 1000 ppm). As leituras altas isoladas continuam na
# série mesmo quando a regra causal as classifica como provável erro de laboratório: uma
# ultrapassagem alta que NÃO terminou em falha é justamente o falso positivo que o modelo
# precisa aprender a rejeitar — e é o mais difícil deles. Ver a seção "Erro de laboratório
# vs medição alta informativa" para o veredito amostra a amostra.
df_crystallizer3, df_descartes_crystallizer3 = limpar_excursoes(
    df_crystallizer3, descartar_isoladas=False
)

tratamentos_crystallizer3, info_crystallizer3 = aplicar_tratamentos(
    df_crystallizer3,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer3

In [ ]:
# Esperado: VAZIO. Nenhuma amostra do C3 passa do teto de 1000 ppm — inclusive as leituras
# de 264, 268/366/488 e 999 ppm ficam na base. O único descarte de toda a base é a amostra
# de 20000 ppm do C2 (df_descartes_crystallizer2).
df_descartes_crystallizer3

### Removendo outliers 0 a 10 #3

In [ ]:
df_crystallizer3_0a10 = tratamentos_crystallizer3["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer3)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer3_0a10)} ({len(df_crystallizer3) - len(df_crystallizer3_0a10)} removidas)")

df_crystallizer3_0a10

### Removendo outliers com IQR #3

In [ ]:
df_crystallizer3_iqr = tratamentos_crystallizer3["IQR"]
limite_inf, limite_sup = info_crystallizer3["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer3)}")
print(f"Amostras removidas: {len(df_crystallizer3) - len(df_crystallizer3_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer3_iqr)}")

df_crystallizer3_iqr

### Removendo outliers com Filtro de Hampel #3

In [ ]:
df_crystallizer3_hampel = tratamentos_crystallizer3["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer3['outliers_hampel']}")
df_crystallizer3_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #3

In [ ]:
df_crystallizer3_hampel90d = tratamentos_crystallizer3["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer3['outliers_hampel_longo']}")
df_crystallizer3_hampel90d

# Erro de laboratório vs medição alta informativa

O corte fixo de 100 ppm que existia antes descartava as duas leituras que a planilha de inspeção registra como **causa** de parada de emergência (264 ppm em 23/11/2013 e 999 ppm em 02/08/2018, ambas no C3, ambas com furo confirmado na abertura).

## Política atual: só o valor EXTREMO é descartado

`limpar_excursoes` remove **uma única classe de amostra**: a que passa do **teto de implausibilidade** (1000 ppm). Na base inteira existe exatamente uma — 20000 ppm no C2, 05/02/2019 — e a segunda maior leitura de toda a base é 999 ppm, confirmada por furo. Todo o resto permanece na série.

A regra causal (`classificar_amostras_altas`) continua rodando, mas agora é **diagnóstico, não filtro**. Ela responde "essa leitura alta tem cara de excursão real ou de leitura isolada?" através de dois critérios:

1. **corroboração** — pelo menos 2 outras amostras acima de 5 ppm em ±2 dias;
2. **seguida de parada** — a próxima amostra normal só aparece mais de 2 dias depois (numa falha abrupta a operação para o reator logo após a leitura, então a excursão não chega a aparecer em outras amostras — foi o caso de 23/11/2013 no C3).

### Por que a leitura alta isolada NÃO é mais descartada

Uma leitura isolada acima de 20 ppm é uma **ultrapassagem que não terminou em falha** — ou seja, é exatamente o falso positivo que o modelo tem de aprender a rejeitar, e é o mais difícil deles, porque é o de maior amplitude. Apagá-la fazia três estragos:

- **retirava da base o exemplo negativo mais informativo** (são 13 amostras entre 21 e 37 ppm, distribuídas em C1 5, C2 2, C3 6);
- **inflava a precisão dos detectores de cauda por construção**: `max > 20 ppm` ia de 14 para 5 falsos positivos e de 0.222 para 0.375 de precisão sem que o detector tivesse melhorado em nada — o ganho vinha de o filtro ter apagado justamente os casos que o detector erra;
- **descartava pelo menos um caso informativo**: a leitura de 23.7 ppm no C1 em 20/05/2011, classificada como isolada, está **8.5 dias antes** de uma falha ancorada. Mantê-la faz `max > 10 ppm` subir de 6 para 7 verdadeiros positivos (F2 0.181 → 0.200).

Efeito de manter tudo (só o teto): **13 amostras a mais** na base, **+4 eventos Real=0** no LC 10 ppm (25 → 29) e **+1** no LC 5 ppm (103 → 104). As 31 falhas ancoradas não mudam.

Robustez a spike espúrio passa a vir de onde deve vir — de **feature robusta** (mediana diária, p75/p90) e das features relativas — e não de apagar amostra. A tabela abaixo mostra o veredito de cada leitura alta; nenhuma delas sai da base.

In [ ]:
# Todas as amostras acima de 20 ppm e o veredito da regra causal.
# ATENÇÃO: o veredito é DIAGNÓSTICO — nenhuma destas amostras é removida da base.
# "isolada" NÃO significa "erro descartado": significa candidata a falso positivo,
# que é precisamente o que a classe Real=0 do modelo precisa conter.
LC_ALTO = 20

relatorio_excursoes = []
for nome, df in [("C1", df_crystallizer1), ("C2", df_crystallizer2), ("C3", df_crystallizer3)]:
    c = classificar_amostras_altas(df, lc_alto=LC_ALTO)
    alta = c[c["Alta"]].copy()
    alta["Crystallizer"] = nome
    relatorio_excursoes.append(alta)

relatorio_excursoes = pd.concat(relatorio_excursoes, ignore_index=True)
relatorio_excursoes["Veredito"] = np.where(
    relatorio_excursoes["ExcursaoReal"], "excursão real", "isolada (candidata a falso positivo)"
)
relatorio_excursoes = relatorio_excursoes.sort_values(["Crystallizer", "TIMESTAMP"])

print(relatorio_excursoes["Veredito"].value_counts().to_string())
print(f"Todas mantidas na base (descarte só acima de 1000 ppm): {len(relatorio_excursoes)}")

relatorio_excursoes[["Crystallizer", "TIMESTAMP", "Resultado de Ferro (ppm)",
                     "Corroborada", "SeguidaDeParada", "Veredito"]]

In [ ]:
# O que REALMENTE saiu da base — só o extremo acima do teto de implausibilidade
for nome, desc in [("C1", df_descartes_crystallizer1),
                   ("C2", df_descartes_crystallizer2),
                   ("C3", df_descartes_crystallizer3)]:
    print(f"{nome}: {len(desc)} amostra(s) descartada(s)")
    if len(desc):
        print(desc[["TIMESTAMP", "Resultado de Ferro (ppm)", "Motivo"]].to_string(index=False))
    print()

# Contraprova da decisão: o que a política antiga (descartar as leituras isoladas) tiraria.
# Cada uma dessas amostras é uma ultrapassagem sem falha — matéria-prima da classe Real=0.
print("-" * 78)
print("Leituras altas isoladas MANTIDAS de propósito (candidatas a falso positivo):")
for nome, df in [("C1", df_crystallizer1), ("C2", df_crystallizer2), ("C3", df_crystallizer3)]:
    c = classificar_amostras_altas(df, lc_alto=LC_ALTO)
    iso = c[c["ErroLab"]]
    faixa = (f"{iso['Resultado de Ferro (ppm)'].min():.1f}-{iso['Resultado de Ferro (ppm)'].max():.1f} ppm"
             if len(iso) else "-")
    print(f"  {nome}: {len(iso):>2} amostra(s)  ({faixa})")

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
# LC usado para sintetizar a classe negativa (Real=0): ultrapassagens sem falha relatada.
# 10 ppm (e não os 5 ppm operacionais) torna o negativo "difícil": uma excursão clara que
# mesmo assim não terminou em falha. A variante no LC operacional de 5 ppm — o cenário real
# de alarme, com ~3x mais negativos — é construída na seção "Unificando eventos".
threshold = 10 #10 #5 #7

## Planilha de inspeção e tabela de eventos

Os eventos vêm de `data/Vitrificados do PIA - Dados de inspeção.xlsx` (uma aba por reator). Três coisas acontecem aqui:

- **reancoragem por lacuna** — quando o reator para, a amostragem para junto, e o apontamento costuma cair no meio do período sem medição. O evento é deslocado para a última medição antes da lacuna, para que a janela `[ts - dias, ts)` cubra os dados que realmente antecedem a parada;
- **parada de planta vs parada de reator** — lacunas que atingem os três reatores ao mesmo tempo são parada de planta ou de laboratório, não falha de equipamento. Eventos que caem numa dessas janelas ficam de fora do conjunto de falhas;
- **fusão de apontamentos** — a planilha registra a mesma parada em mais de uma linha; apontamentos do mesmo reator a menos de 7 dias viram um evento só.

In [ ]:
MAP_MEDICOES_BASE = {
    "C1": df_crystallizer1,
    "C2": df_crystallizer2,
    "C3": df_crystallizer3,
}

df_inspecoes_bruto = carregar_inspecoes()
paradas_de_planta  = identificar_paradas_de_planta(MAP_MEDICOES_BASE)

print(f"Paradas de planta identificadas (lacuna simultânea nos 3 reatores): {len(paradas_de_planta)}")
for inicio, fim in paradas_de_planta:
    print(f"   {inicio:%d/%m/%Y} -> {fim:%d/%m/%Y}  ({(fim - inicio).days} dias)")

In [ ]:
df_inspecoes = montar_tabela_eventos(df_inspecoes_bruto, MAP_MEDICOES_BASE, paradas_de_planta)

In [ ]:
# Os apontamentos que foram reancorados e o quanto andaram
deslocados = df_inspecoes[df_inspecoes["NoPeriodo"] & df_inspecoes["Deslocado"]]
deslocados[["Crystallizer", "Inicio", "TS_Ajustado", "DiasDeslocado", "LacunaDias",
            "LacunaDePlanta", "Selecionado", "Ocorrimento"]].sort_values(["Crystallizer", "Inicio"])

## Crystallizer #1

In [ ]:
df_eventos_crystallizer1 = eventos_para_notebook(df_inspecoes, "C1")

print(f"Falhas de reator no C1: {len(df_eventos_crystallizer1)}")
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem LC
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer1 = adicionar_eventos_ultrapassagem(
    df_crystallizer1, df_eventos_crystallizer1, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer1

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer1},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer1_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer1, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #1"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#1.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer1_filtrado_until2020, df_eventos_crystallizer1_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer1, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer1_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer1_filtrado_after2020)} eventos")

df_eventos_crystallizer1_filtrado_until2020.head()

## Crystallizer #2

In [ ]:
df_eventos_crystallizer2 = eventos_para_notebook(df_inspecoes, "C2")

print(f"Falhas de reator no C2: {len(df_eventos_crystallizer2)}")
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer2 = adicionar_eventos_ultrapassagem(
    df_crystallizer2, df_eventos_crystallizer2, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer2

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer2},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer2_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer2, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #2"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#2.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer2_filtrado_until2020, df_eventos_crystallizer2_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer2, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer2_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer2_filtrado_after2020)} eventos")

df_eventos_crystallizer2_filtrado_until2020.head()

## Crystallizer #3

In [ ]:
df_eventos_crystallizer3 = eventos_para_notebook(df_inspecoes, "C3")

print(f"Falhas de reator no C3: {len(df_eventos_crystallizer3)}")
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer3 = adicionar_eventos_ultrapassagem(
    df_crystallizer3, df_eventos_crystallizer3, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer3

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer3},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer3_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer3, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #3"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#3.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer3_filtrado_until2020, df_eventos_crystallizer3_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer3, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer3_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer3_filtrado_after2020)} eventos")

df_eventos_crystallizer3_filtrado_until2020.head()

# Baseline: o que a regra vigente entrega

A Etapa 2 tem como objetivo superar o modelo univariado em uso (patamar fixo de ferro). Para afirmar que algo o supera é preciso primeiro medi-lo — e medir contra a **taxa base**, não contra zero.

O lift compara a fração de janelas que antecedem falha atendendo a um critério com a mesma fração em janelas sorteadas ao acaso. Lift próximo de 1 significa que o critério dispara tanto antes de falha quanto em qualquer outro momento — ou seja, não informa nada.

In [ ]:
MAP_EVENTOS_FALHA = {
    "C1": df_eventos_crystallizer1[df_eventos_crystallizer1["Real"] == 1],
    "C2": df_eventos_crystallizer2[df_eventos_crystallizer2["Real"] == 1],
    "C3": df_eventos_crystallizer3[df_eventos_crystallizer3["Real"] == 1],
}

def formatar_lift(df):
    saida = df.copy()
    saida["nas_falhas"] = (100 * saida["nas_falhas"]).round(1).astype(str) + "%"
    saida["taxa_base"]  = (100 * saida["taxa_base"]).round(1).astype(str) + "%"
    saida["lift"]       = saida["lift"].round(2).astype(str) + "x"
    return saida

df_lift = avaliar_baseline_lift(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, janela_dias=15)
formatar_lift(df_lift)

In [ ]:
# Mesma comparação com janela de 60 dias — o limite de 5 ppm fica ABAIXO da taxa base
df_lift_60 = avaliar_baseline_lift(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, janela_dias=60)
formatar_lift(df_lift_60)

## Detector: precisão, recall e F2 da regra vigente e das alternativas

Alarme = estatística diária acima do limite; alarmes a menos de 15 dias contam como um só; acerto se a falha ocorre em até 15 dias depois do alarme.

A tabela abaixo cobre só critérios de patamar sobre a série de cada reator. A régua final — incluindo a **regra composta** com a margem cross-reator, que é a que a Etapa supervisionada precisa bater — está em "Estrutura não supervisionada / A nova régua", depois que a margem é construída.

In [ ]:
configuracoes = [
    ("max",    5,   1),   # regra vigente
    ("max",    7,   1),
    ("max",    10,  1),
    ("max",    20,  1),
    ("median", 3.0, 1),
    ("median", 3.5, 1),
    ("median", 4.0, 1),
    ("median", 3.5, 2),   # exigindo 2 dias seguidos — mostra que o sinal é impulso, não rampa
]

df_detector = pd.DataFrame([
    avaliar_detector_diario(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
                            estatistica=est, limite=lim, dias_seguidos=dias)
    for est, lim, dias in configuracoes
]).sort_values("F2", ascending=False)

df_detector

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

## Crystallizer #1

In [ ]:
# Sem eventos falsos
fig_crystallizer1 = plot_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1",
    mostrar_falsos=False
)
fig_crystallizer1.show()

In [ ]:
# fig_crystallizer1.write_html("Crystallizer#1.html")

### Crystallizer #1 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer1_0a10 = plot_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 0a10",
    mostrar_falsos=False
)
fig_crystallizer1_0a10.show()

### Crystallizer #1 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer1_iqr = plot_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 IQR",
    mostrar_falsos=False
)
fig_crystallizer1_iqr.show()

### Crystallizer #1 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer1_hampel = plot_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 Hampel",
    mostrar_falsos=False
)
fig_crystallizer1_hampel.show()

## Crystallizer #2

In [ ]:
# Sem eventos falsos
fig_crystallizer2 = plot_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2",
    mostrar_falsos=False
)
fig_crystallizer2.show()

In [ ]:
# fig_crystallizer2.write_html("Crystallizer#2.html")

### Crystallizer #2 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer2_0a10 = plot_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 0a10",
    mostrar_falsos=False
)
fig_crystallizer2_0a10.show()

### Crystallizer #2 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer2_iqr = plot_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 IQR",
    mostrar_falsos=False
)
fig_crystallizer2_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer2_hampel = plot_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 Hampel",
    mostrar_falsos=False
)
fig_crystallizer2_hampel.show()

## Crystallizer #3

In [ ]:
# Sem eventos falsos
fig_crystallizer3 = plot_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3",
    mostrar_falsos=False
)
fig_crystallizer3.show()

In [ ]:
# fig_crystallizer3.write_html("Crystallizer#3.html")

### Crystallizer #3 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer3_0a10 = plot_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 0a10",
    mostrar_falsos=False
)
fig_crystallizer3_0a10.show()

### Crystallizer #3 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer3_iqr = plot_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 IQR",
    mostrar_falsos=False
)
fig_crystallizer3_iqr.show()

### Crystallizer #3 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer3_hampel = plot_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 Hampel",
    mostrar_falsos=False
)
fig_crystallizer3_hampel.show()

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

## MM Crystallizer #1

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #2

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #3

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 Hampel",
    mostrar_falsos=False
)
fig.show()

# Estatísticas descritivas de toda série de concentração de Fe

In [ ]:
# Coluna separadora vazia
separador = pd.Series({k: "" for k in ["Contagem","Média","Mediana","Desvio Padrão","Variância",
                                        "Mínimo","Máximo","Amplitude","Q1 (25%)","Q3 (75%)","IQR",
                                        "Assimetria","Curtose"]})

c1 = pd.concat([
    estatisticas(df_crystallizer1["Resultado de Ferro (ppm)"].dropna(),       "C1 Original"),
    estatisticas(df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna(),  "C1 Intervalo 0-10"),
    estatisticas(df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna(),   "C1 IQR"),
    estatisticas(df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna(),"C1 Hampel"),
], axis=1)

c2 = pd.concat([
    estatisticas(df_crystallizer2["Resultado de Ferro (ppm)"].dropna(),       "C2 Original"),
    estatisticas(df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna(),  "C2 Intervalo 0-10"),
    estatisticas(df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna(),   "C2 IQR"),
    estatisticas(df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna(),"C2 Hampel"),
], axis=1)

c3 = pd.concat([
    estatisticas(df_crystallizer3["Resultado de Ferro (ppm)"].dropna(),       "C3 Original"),
    estatisticas(df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna(),  "C3 Intervalo 0-10"),
    estatisticas(df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna(),   "C3 IQR"),
    estatisticas(df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna(),"C3 Hampel"),
], axis=1)

sep = separador.rename("│")

df_comparativo = pd.concat([c1, sep, c2, sep.rename("│"), c3], axis=1).round(4)

# Corrige as colunas separadoras que ficaram com float após o round
df_comparativo["│"]  = ""
df_comparativo["│"] = ""

df_comparativo

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

CORES = {"C1": "#1f77b4", "C2": "#2ca02c", "C3": "#ff7f0e"}

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=2,
    subplot_titles=[
        titulo
        for m in metodos
        for titulo in [f"KDE — {m['titulo']}", f"Violin Plot — {m['titulo']}"]
    ]
)

for row_idx, metodo in enumerate(metodos, start=1):
    for c in metodo["series"]:
        serie = c["serie"]
        nome  = c["nome"]
        cor   = CORES[nome]

        # KDE na escala de densidade natural (área sob a curva = 1)
        kde = gaussian_kde(serie)
        x_range = np.linspace(serie.min(), serie.max(), 1000)
        y_kde = kde(x_range)
        y_kde = y_kde / trapezoid(y_kde, x_range)  # normaliza

        fig.add_trace(go.Scatter(
            x=x_range,
            y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=hex_to_rgba(cor, alpha=0.15),
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Violin
        fig.add_trace(go.Violin(
            y=serie,
            name=nome,
            marker_color=cor,
            fillcolor=hex_to_rgba(cor, alpha=0.4),
            box_visible=True,
            meanline_visible=True,
            legendgroup=nome,
            showlegend=False
        ), row=row_idx, col=2)

    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_yaxes(title_text="ppm", row=row_idx, col=2)

fig.update_layout(
    height=500 * n_metodos,
    template='plotly_white',
    title="Análise Descritiva Comparativa — Resultado de Ferro (ppm)",
)
fig.show()

### Teste de Kruskal-Wallis com Effect Size (η²)

Para amostras grandes (n ~ 30.000), testes estatísticos como o Kruskal-Wallis tendem a rejeitar a hipótese nula mesmo com diferenças praticamente irrelevantes. Por isso o p-value é complementado pelo **eta-quadrado (η²)** que mede a proporção da variação total explicada pelo agrupamento por crystallizer.

| η²        | Interpretação                                      |
|-----------|----------------------------------------------------|
| < 0.01    | Efeito negligenciável — unificação justificada     |
| 0.01–0.06 | Efeito pequeno — unificação provavelmente aceitável|
| 0.06–0.14 | Efeito médio — avaliar com cautela                 |
| > 0.14    | Efeito grande — distribuições substancialmente diferentes |

A decisão de unificar os dados dos três crystallizers deve considerar em conjunto o η², o p-value e a inspeção visual das curvas KDE e violin plots gerados.

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

for metodo in metodos:
    series = [c["serie"].values for c in metodo["series"]]
    nomes  = [c["nome"] for c in metodo["series"]]
    n_total = sum(len(s) for s in series)

    stat_kw, p_kw = kruskal(*series)

    # Eta-quadrado: mede o quanto da variação total é explicada pelo grupo
    # 0.01 = pequeno, 0.06 = médio, 0.14 = grande
    eta2 = (stat_kw - len(series) + 1) / (n_total - len(series))

    print(f"\nMétodo: {metodo['titulo']}")
    print(f"  H = {stat_kw:.4f}  |  p = {p_kw:.6f}  |  η² = {eta2:.4f}")
    if eta2 < 0.01:
        print("  → Efeito negligenciável — unificação justificada mesmo com p < 0.05")
    elif eta2 < 0.06:
        print("  → Efeito pequeno — unificação provavelmente aceitável")
    elif eta2 < 0.14:
        print("  → Efeito médio — avaliar com cautela")
    else:
        print("  → Efeito grande — distribuições substancialmente diferentes")

### Teste de normalidade Q-Q Plot

In [ ]:
# Dataset: Original — testar normalidade sobre a série IQR seria circular (o IQR amputa as
# caudas e a conclusão sai enviesada para "normal"). Ver "Política de tratamento de dados".
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

serie = df["Resultado de Ferro (ppm)"].dropna()

# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie.min(), serie.max(), 300)
y_normal = stats.norm.pdf(x_range, serie.mean(), serie.std())
y_normal_scaled = y_normal * len(serie) * (serie.max() - serie.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

# Unificando bases de dados

## Unificando dados

In [ ]:
# Original
df_crystallizer123 = pd.concat([
    df_crystallizer1.assign(Crystallizer='C1'),
    df_crystallizer2.assign(Crystallizer='C2'),
    df_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Originais: {df_crystallizer123.shape}")

# Intervalo 0-10
df_crystallizer123_0a10 = pd.concat([
    df_crystallizer1_0a10.assign(Crystallizer='C1'),
    df_crystallizer2_0a10.assign(Crystallizer='C2'),
    df_crystallizer3_0a10.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Intervalo 0-10: {df_crystallizer123_0a10.shape}")

# IQR
df_crystallizer123_iqr = pd.concat([
    df_crystallizer1_iqr.assign(Crystallizer='C1'),
    df_crystallizer2_iqr.assign(Crystallizer='C2'),
    df_crystallizer3_iqr.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"IQR: {df_crystallizer123_iqr.shape}")

# Hampel
df_crystallizer123_hampel = pd.concat([
    df_crystallizer1_hampel.assign(Crystallizer='C1'),
    df_crystallizer2_hampel.assign(Crystallizer='C2'),
    df_crystallizer3_hampel.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Hampel: {df_crystallizer123_hampel.shape}")

## Unificando eventos

In [ ]:
# Eventos unificados — igual para todos os tratamentos.
# A deduplicação cross-reator (remoção de Real=0 perto de qualquer Real=1 e espaçamento
# mínimo entre Real=0) vive em utils.unificar_eventos, usada também pela variante de
# 5 ppm abaixo — as duas tabelas passam exatamente pela mesma regra.
df_eventos_crystallizer123 = unificar_eventos({
    "C1": df_eventos_crystallizer1,
    "C2": df_eventos_crystallizer2,
    "C3": df_eventos_crystallizer3,
}, intervalo_min_dias=15)

## Variante: classe negativa no LC operacional (5 ppm)

O `threshold = 10` gera negativos "difíceis" (excursões claras sem falha), mas a regra vigente
e o custo da condenação indevida vivem em **5 ppm**. Para a Etapa 2 responder "reduzimos os
falsos positivos da regra vigente?", a classe negativa também precisa existir no limiar em que
a planta opera — o que multiplica os negativos e torna o problema mais realista.

As duas tabelas seguem em paralelo: `df_eventos_*` (LC 10 ppm) e `df_eventos_*_lc5` (LC 5 ppm).
Qualquer resultado supervisionado deve ser reportado nas duas.

In [ ]:
LC_OPERACIONAL = 5

# Reconstrói a parte Real=1 direto da planilha de inspeção (eventos_para_notebook) em vez de
# reaproveitar df_eventos_crystallizerN — evita herdar os Real=0 de LC 10 e o risco de
# duplicação por reexecução.
df_eventos_crystallizer1_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer1, eventos_para_notebook(df_inspecoes, "C1"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)
df_eventos_crystallizer2_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer2, eventos_para_notebook(df_inspecoes, "C2"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)
df_eventos_crystallizer3_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer3, eventos_para_notebook(df_inspecoes, "C3"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)

df_eventos_crystallizer123_lc5 = unificar_eventos({
    "C1": df_eventos_crystallizer1_lc5,
    "C2": df_eventos_crystallizer2_lc5,
    "C3": df_eventos_crystallizer3_lc5,
}, intervalo_min_dias=15)

# Comparação das duas classes negativas
comparacao = pd.DataFrame({
    "LC 10 ppm": df_eventos_crystallizer123["Real"].value_counts(),
    "LC 5 ppm (operacional)": df_eventos_crystallizer123_lc5["Real"].value_counts(),
}).rename(index={1: "Real=1 (falhas)", 0: "Real=0 (ultrapassagens)"})
comparacao

## Auditoria da tabela de eventos e ficha para a planta

Duas verificações que faltavam antes de qualquer modelo consumir esta tabela:

1. **suporte da janela** — nas janelas deslizantes o suporte insuficiente é filtrado, mas na
   tabela de eventos um evento com poucas amostras em 15 dias entra calado e vira estatística
   de dois pontos;
2. **ficha de validação** — todo resultado que depende de rótulo repousa nestes timestamps, e
   há divergência conhecida entre o deck da equipe Bayer e a planilha de inspeção em pelo
   menos três eventos. A ficha põe lado a lado data da planilha, data reancorada, deslocamento
   aplicado, motivo e suporte de amostra, **para a planta validar linha a linha**.

In [ ]:
MAP_FALHAS_AUDITORIA = {
    "C1": df_eventos_crystallizer1[df_eventos_crystallizer1["Real"] == 1],
    "C2": df_eventos_crystallizer2[df_eventos_crystallizer2["Real"] == 1],
    "C3": df_eventos_crystallizer3[df_eventos_crystallizer3["Real"] == 1],
}

df_auditoria_janelas = auditar_janelas_eventos(MAP_MEDICOES_BASE, MAP_FALHAS_AUDITORIA)

print()
df_ficha_eventos = ficha_eventos_para_validacao(df_inspecoes, MAP_MEDICOES_BASE)
print(f"Ficha para validação com a planta: {len(df_ficha_eventos)} apontamentos no período")
print(df_ficha_eventos["status"].value_counts().to_string())
# df_ficha_eventos.to_csv(DIR_OUTPUT + "ficha_eventos_para_validacao.csv", sep=";", index=False)
df_ficha_eventos.head(10)

## Violin Plot das classes

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer123},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer123_0a10},
    {"titulo": "IQR",            "df": df_crystallizer123_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer123_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer123, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1#2#3 unificado"
)
fig.show()

## Plot dados unificados

In [ ]:
# Sem eventos falsos
fig = plot_crystallizer_unificado(
    df_crystallizer123, df_eventos_crystallizer123,
    titulo="Fe (ppm) - Crystallizers Unificados (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer1, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer1, JANELAS, titulo="Crystallizer #1"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 — eventos após 01/04/2020"
)
fig.show()

## Crystallizer #2

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
# df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer2, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
# df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer2, JANELAS, titulo="Crystallizer #2"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 — eventos após 01/04/2020"
)
fig.show()

## Crystallizer #3

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
# df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer3, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
# df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer3, JANELAS, titulo="Crystallizer #3"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 — eventos após 01/04/2020"
)
fig.show()

# Comparação dos 3 reatores

## Verificando effect size por reator

In [ ]:
crystallizers_config = [
    # C1
    (df_crystallizer1,        df_eventos_crystallizer1, "C1 Original"),
    (df_crystallizer1_0a10,   df_eventos_crystallizer1, "C1 0-10"),
    (df_crystallizer1_iqr,    df_eventos_crystallizer1, "C1 IQR"),
    (df_crystallizer1_hampel, df_eventos_crystallizer1, "C1 Hampel"),
    # C2
    (df_crystallizer2,        df_eventos_crystallizer2, "C2 Original"),
    (df_crystallizer2_0a10,   df_eventos_crystallizer2, "C2 0-10"),
    (df_crystallizer2_iqr,    df_eventos_crystallizer2, "C2 IQR"),
    (df_crystallizer2_hampel, df_eventos_crystallizer2, "C2 Hampel"),
    # C3
    (df_crystallizer3,        df_eventos_crystallizer3, "C3 Original"),
    (df_crystallizer3_0a10,   df_eventos_crystallizer3, "C3 0-10"),
    (df_crystallizer3_iqr,    df_eventos_crystallizer3, "C3 IQR"),
    (df_crystallizer3_hampel, df_eventos_crystallizer3, "C3 Hampel"),
    # Unificado
    (df_crystallizer123,        df_eventos_crystallizer123, "Unificado Original"),
    (df_crystallizer123_0a10,   df_eventos_crystallizer123, "Unificado 0-10"),
    (df_crystallizer123_iqr,    df_eventos_crystallizer123, "Unificado IQR"),
    (df_crystallizer123_hampel, df_eventos_crystallizer123, "Unificado Hampel"),
]

STATS_FUNCS = {
    'media'   : np.mean,
    'mediana' : np.median,
    'std'     : np.std,
    'max'     : np.max,
    'p75'     : lambda x: np.percentile(x, 75),
    'p90'     : lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range'   : lambda x: np.max(x) - np.min(x),
}

registros_todos = []
for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v      = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela'      : f"{DIAS_JANELA}d",
                'feature'     : nome_stat,
                'effect_size' : round(effect, 4),
                'p_value'     : round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap — 4 origens (C1, C2, C3, Unificado) × 4 métodos = 16 subplots
n_cols = 4  # Original, 0-10, IQR, Hampel
n_rows = 4  # C1, C2, C3, Unificado
nomes  = [c[2] for c in crystallizers_config]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=nomes,
    vertical_spacing=0.06
)

for idx, (_, _, nome_c) in enumerate(crystallizers_config):
    row_idx = idx // n_cols + 1
    col_idx = idx % n_cols + 1

    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(col_idx == n_cols and row_idx == n_rows)
    ), row=row_idx, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — C1, C2, C3 e Unificado × Original, 0-10, IQR, Hampel",
    template="plotly_white",
    height=400 * n_rows
)
fig.show()

## Correlação cruzada entre os crystallizers

In [ ]:
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

    print(f"\n{'='*55}")
    print(f"Método: {m['titulo']}")
    print(m["df_corr"].corr(method="pearson").round(3).to_string())

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=3,
    subplot_titles=[
        f"{a} vs {b} — {m['titulo']}"
        for m in metodos
        for a, b in pares
    ],
    vertical_spacing=0.06
)

for row_idx, m in enumerate(metodos, start=1):
    for col_idx, (a, b) in enumerate(pares, start=1):
        add_scatter_regressao(fig, m["df_corr"][a].values, m["df_corr"][b].values, row=row_idx, col=col_idx)
        fig.update_xaxes(title_text=a, row=row_idx, col=col_idx)
        fig.update_yaxes(title_text=b, row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template='plotly_white',
    title="Correlação cruzada — C1, C2, C3 × Original, 0-10, IQR, Hampel"
)
fig.show()

## Correlação cruzada com lags
Verificando se um reator tem influência sobre outro

In [ ]:
MAX_LAG = 15
lags = range(-MAX_LAG, MAX_LAG + 1)
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

# Prepara df_corr para cada método
for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

# Cross-correlação com lag
n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Cross-correlação com Defasagem — {m['titulo']}" for m in metodos],
    vertical_spacing=0.08
)

for row_idx, m in enumerate(metodos, start=1):
    df_c = m["df_corr"]
    resultados_lag = []

    for a, b in pares:
        x = df_c[a].values
        y = df_c[b].values
        for lag in lags:
            if lag < 0:
                xs, ys = x[:lag],  y[-lag:]
            elif lag > 0:
                xs, ys = x[lag:],  y[:-lag]
            else:
                xs, ys = x, y
            r, p = pearsonr(xs, ys)
            resultados_lag.append({
                "par": f"{a} vs {b}", "lag_dias": lag * 3,
                "r": round(r, 4), "p": round(p, 6)
            })

    df_lag = pd.DataFrame(resultados_lag)

    for par in df_lag["par"].unique():
        sub = df_lag[df_lag["par"] == par]
        fig.add_trace(go.Scatter(
            x=sub["lag_dias"], y=sub["r"],
            mode="lines", name=par,
            line=dict(width=2),
            legendgroup=par,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.add_hline(y=0, line_color="gray", line_width=0.5, row=row_idx, col=1)
    fig.update_yaxes(title_text="Pearson r", row=row_idx, col=1)
    fig.update_xaxes(title_text="Defasagem (dias)", row=row_idx, col=1)

    # Lag de máxima correlação
    print(f"\nMétodo: {m['titulo']} — Lag de máxima correlação:")
    for par in df_lag["par"].unique():
        sub  = df_lag[df_lag["par"] == par]
        best = sub.loc[sub["r"].idxmax()]
        print(f"  {par}: lag={best['lag_dias']:.0f} dias  |  r={best['r']:.4f}")

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    hovermode="x unified",
    title="Correlação cruzada com Defasagem entre Reatores<br>"
          "<sup>Pico em lag≠0 indica que um reator influencia o outro, negativo: A influencia B  |  positivo: B influencia A</sup>"
)
fig.show()

# Estrutura não supervisionada (regimes de operação)

1. **Janelas deslizantes de toda a série** (~5000 janelas, uma a cada 3 dias) em vez das 59 linhas
   de evento — só assim existe "regime de operação" a descobrir;
2. **log1p + `RobustScaler`**, porque a série é fortemente assimétrica e sem isso a silhueta é
   sequestrada por meia dúzia de excursões;
3. **`k` escolhido pela silhueta** (critério interno). O rótulo entra apenas *depois*, para medir
   o **lift** de cada regime — a mesma régua do Baseline;
4. **Decomposição cross-reator**, **idade de campanha** e **qualidade de janela**, que são as
   dimensões que a série univariada sozinha não tem.

## Insumos: um contexto único para features e modelos

`preparar_contexto` monta de uma vez todas as séries auxiliares e devolve um dicionário —
passar um único `contexto` em vez de meia dúzia de mapas evita o erro de calcular uma feature
com um recorte e outra com outro. O que entra nele:

| Item | O que é | Por que existe |
|---|---|---|
| `margem`, `posto` | mediana diária do reator menos a mediana dos três; posição do reator no dia | com três séries a mediana é o valor do meio, então a margem é positiva só para o reator que lidera. Lift 5.3x–6.9x contra 3.6x do nível absoluto |
| `baseline` | quantil 0.99 móvel de 365 d de cada reator | o drift (mediana anual ~2.7 → ~1.8 ppm) faz um limiar fixo significar coisas diferentes em 2013 e 2025 |
| `controle` | EWMA (baseline rolante) e CUSUM dinâmico, amostra a amostra | as cartas viravam gráfico e paravam ali; aqui a estatística no instante da âncora vira **feature** — as duas acumulam desvio pequeno e persistente, que `max` e mediana ignoram |
| `campanhas`, `reparos`, `falhas` | histórico do equipamento pela planilha de inspeção | idade de campanha, reparos desde a última troca e falhas anteriores: informação que a série de ferro não tem como conter |

In [ ]:
CONTEXTO = preparar_contexto(MAP_MEDICOES_BASE, df_inspecoes)

# Atalhos usados nas células seguintes (tudo vem do mesmo contexto, de propósito)
diario_mediana      = CONTEXTO["diario"]
comum_planta        = CONTEXTO["comum"]
margem_cross_reator = CONTEXTO["margem"]
MAP_MARGEM          = {c: CONTEXTO["margem"][c] for c in MAP_MEDICOES_BASE}
CAMPANHAS           = CONTEXTO["campanhas"]

print("\nTrocas de reator conhecidas:", {c: len(v) for c, v in CAMPANHAS.items()})

## Janelas deslizantes

Uma âncora a cada 3 dias, janela de 15 dias, contexto de 90 dias. Janelas sem suporte
(menos de 8 amostras) são **descartadas explicitamente** em vez de virarem estatística de 2 pontos —
esse era um viés silencioso: os eventos variavam de 2 a 150 amostras na janela.

`pre_falha` marca as janelas seguidas de falha em até 15 dias. Ele **não** entra em nenhum ajuste,
só na avaliação.

In [ ]:
df_janelas = gerar_janelas_deslizantes(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    passo_dias=3, janela_dias=15, janela_base_dias=90,
    contexto=CONTEXTO
)

# Mesmas features calculadas exatamente nas 31 âncoras de falha
df_janelas_falha = janelas_nas_falhas(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, contexto=CONTEXTO)

print(f"\nJanelas nas âncoras de falha: {len(df_janelas_falha)}")
print(f"Features por janela: {len([c for c in COLUNAS_MODELO_JANELA if c in df_janelas.columns])}")
df_janelas.head()

## Regimes de operação

`k` escolhido pela silhueta. A tabela do grid mostra também, para cada `k`, o melhor lift
alcançável — inclusive o `lift_util`, restrito a regimes que cobrem pelo menos 20% das falhas.
Sem essa coluna, um cluster com 5 janelas e lift 27x parece um achado; com ela fica claro que ele
cobre 1 falha de 31.

In [ ]:
res_regimes = clusterizar_regimes(df_janelas, df_janelas_falha)

In [ ]:
fig_regimes = plotar_regimes_pca(res_regimes, df_janelas_falha,
                                titulo="Regimes de operação — janelas de 15 dias")
fig_regimes.show()

### Conclusão dos regimes

**Não existe regime pré-falha com assinatura própria.** A tabela do grid mostra isso em três níveis:

- o `k` escolhido pela silhueta (k=2) é **degenerado**: 99.6% das janelas em um cluster, com
  **lift 0.97** — ou seja, exatamente a taxa base. Pelo critério interno, os dados não têm grupos;
- em todo `k` testado, o cluster de lift alto é minúsculo (0.1% a 0.4% do tempo) e cobre
  **1 falha de 31**. Aumentando `k` o lift sobe (8.9 → 26.7) e a cobertura **não sai de 3.2%**:
  o algoritmo está isolando as excursões extremas, isto é, **redescobrindo o limiar**;
- o único regime com alguma utilidade operacional aparece em **k=4**: lift **3.27**, cobrindo
  **22.6% das falhas em 6.9% do tempo** (coluna `lift_util`). Só que a assinatura dele é
  simplesmente *ferro alto* (máximo médio de 19 ppm, 8.3x a base de 90 dias) — um limiar disfarçado
  de cluster. E ainda assim rende **menos** que o critério de margem cross-reator da subseção
  seguinte (lift 5.3x a 6.9x), que é uma variável nova, não um recorte da mesma.

Coerente com o Baseline: nos limiares confiáveis o lead time mediano é **0 dias** — o sinal é
impulso, não rampa. Um agrupamento de estatísticas da própria série não tem como separar o que a
série não distingue.

**Consequência prática:** clusterização não é caminho para a etapa supervisionada — nem sobre
eventos (removida), nem sobre janelas (mantida aqui como evidência). O ganho tem que vir de
**dimensões novas**, que é o que as três subseções abaixo trazem.

## Decomposição cross-reator: o que é da planta e o que é do reator

A matriz de correlação acima mostra um fato que muda a leitura do problema: **C1 e C2 andam juntos
(0.76), mas o C3 é quase independente (0.11)** — ele é de outro trem. E o componente comum responde
por apenas ~6% da variância: "todos subiram juntos" é a exceção, não a regra.

A pergunta que interessa é se a **margem** (o quanto este reator está acima dos outros) informa mais
que o **nível absoluto**. Comparação na mesma régua — âncoras de falha contra as ~5000 âncoras de base:

In [ ]:
lift_absoluto = avaliar_lift_series(
    {c: diario_mediana[c] for c in MAP_MEDICOES_BASE},
    df_janelas, df_janelas_falha, [3.0, 3.5, 4.0, 5.0], rotulo="mediana diária absoluta"
)
lift_margem = avaliar_lift_series(
    MAP_MARGEM, df_janelas, df_janelas_falha, [0.5, 0.8, 1.2, 1.6, 2.0], rotulo="margem cross-reator"
)

pd.concat([lift_absoluto, lift_margem], ignore_index=True)

### Conclusão do cross-reator

A margem entrega **lift 5.3x (>1.2) e 6.9x (>1.6)** contra **3.7x** do melhor critério absoluto
(mediana diária > 5 ppm), com taxa base **igual ou menor**. Traduzindo: mesma quantidade de alarme,
quase o dobro de detecção.

Um cuidado medido e descartado: o filtro **binário** de simultaneidade ("ignorar a ultrapassagem se
outro reator também passou") **piora** o detector — a precisão cai de 0.045 para 0.035. É a versão
**contínua** (o quanto este reator lidera) que carrega informação, não o "está sozinho ou não".

## Idade de campanha

O tempo desde a última troca do reator é uma variável que o repositório já tem e que ninguém estava
usando. O ponto metodológico: **normalizar pela exposição**. Contar só as falhas sugere "quanto mais
velho, pior"; dividindo pelo tempo que cada reator passou em cada faixa, o risco tem outro formato.

In [ ]:
perfil_hazard_campanha(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, CAMPANHAS)

### Conclusão da campanha

O risco **tem pico em 1-2 anos** (3.9 falhas por 1000 reator-dias) e cai para 0.7 acima de 5 anos —
5.4x de diferença, e **não monotônico**. Por isso a variável entra no modelo como **faixa ordinal**
(`faixa_campanha`), não como número de dias em regressão linear.

Ressalva: são 12 trocas conhecidas na planilha, e campanhas longas sobreviventes carregam viés de
sobrevivência (o reator que dura 5 anos é, por definição, o que não falhou antes).

## O confundidor que atravessa tudo: as falhas estão no passado

Antes de olhar qualquer feature, é preciso olhar **quando** as falhas acontecem. Se elas se
concentram em um período, qualquer variável que também mude com o tempo vira preditora
espúria — e a validação temporal fica sem poder, porque testar no futuro significa testar
com pouquíssimos positivos.

In [ ]:
perfil_temporal_falhas(MAP_EVENTOS_FALHA, MAP_MEDICOES_BASE)

print()
df_tendencia = testar_tendencia(MAP_MEDICOES_BASE)

### O que essas duas tabelas impõem ao resto do estudo

**28 das 31 falhas (90%) acontecem até 2018.** A taxa cai de ~3.9 falhas por 1000 reator-dias
em 2016-2018 para **0.3** em 2019-2021. Ao mesmo tempo, a série tem tendência de queda
confirmada (tau de Kendall ≈ **-0.29** com p astronômico nos três reatores; inclinação de Sen
≈ **-0.05 ppm por ano**; mediana de 2.45-2.50 ppm em 2011-2015 contra 1.80-1.90 em 2021-2026).

Ou seja: **ferro alto e falha caem juntos ao longo do tempo, sem que um cause o outro.**
Qualquer feature de nível absoluto vai parecer preditiva só por ser mais alta no passado.
É por isso que a tabela seguinte reporta a **AUC ajustada por época** ao lado da AUC bruta — e
é por isso que a série não estacionária justifica limite móvel e features relativas.

## Valor individual de cada feature

AUC de Mann-Whitney de cada feature contra `pre_falha`, medida nas ~4969 janelas (e não nos
~60 eventos, onde não há amostra para a medida significar algo). `AUC_ajustada` recalcula a
separação **dentro de blocos de 3 anos** e faz a média ponderada pelos positivos: quando a AUC
bruta se afasta de 0.5 e a ajustada volta para perto, a feature era um relógio disfarçado.

In [ ]:
df_valor_features = avaliar_valor_features(df_janelas)
df_valor_features

### Conclusão — e ela inverte o senso comum do estudo

| Feature | AUC bruta | AUC ajustada | Leitura |
|---|---|---|---|
| `mediana` | 0.678 | **0.524** | a "melhor feature" era quase toda efeito de época |
| `p90` | 0.655 | **0.541** | idem |
| `razao_max` | 0.384 | **0.487** | a inversão aparente era o drift, não sinal |
| `idade_campanha` | 0.448 | **0.588** | **inverte de sinal e vira a mais forte**: dentro da mesma época, campanha mais velha antecede falha |
| `faixa_campanha` | 0.451 | **0.586** | idem |
| `posto_medio` | 0.418 | **0.416** | estável: quem lidera entre os três reatores tem mais falha |
| `frac_lider` | 0.571 | **0.572** | estável |
| `densidade_relativa` / `n_amostras` | 0.395 / 0.396 | 0.421 / 0.431 | menos amostras antes da falha: a amostragem cai perto da parada |
| `cusum_rel` | 0.562 | **0.567** | estável — a carta de controle como feature carrega sinal próprio |
| `margem_media` | 0.568 | **0.565** | estável |
| `n_reparos_vidro_campanha` | 0.449 | **0.559** | também inverte com o ajuste: mais reparos de vidro na campanha, mais risco |
| `n_reparos_na_campanha` | 0.453 | 0.553 | idem, versão que conta qualquer reparo |

Três consequências práticas:

1. **Nível absoluto de ferro é o pior tipo de feature aqui** — não porque não separe, mas porque
   o que ele separa é o ano. Confirma, por outro caminho, o diagnóstico do deck da Bayer, e
   explica por que um limiar fixo envelhece mal.
2. **O que sobrevive ao ajuste é o que não é nível**: idade de campanha, histórico de reparo do
   revestimento vitrificado, posição relativa entre reatores, acúmulo do CUSUM e densidade de
   amostragem. São exatamente as dimensões que a série univariada sozinha não tinha — e as três
   primeiras descrevem **degradação do equipamento**, que é a física do problema.
3. **Nenhuma feature isolada passa de AUC ajustada 0.59.** Não existe variável salvadora neste
   conjunto; o ganho, se houver, vem de combinação — e precisa ser demonstrado com validação
   temporal, não com um split único.

Ressalva sobre a margem: `margem_max` tem AUC ajustada de apenas 0.519, mas lift de **6.9x** no
limiar de 1.6. Não é contradição — AUC é medida de separação **média**, lift é medida de **cauda**.
A margem não distingue o dia comum; distingue o dia extremo, que é o que interessa para alarme.

**Features consideradas e recusadas de propósito:** ano ou indicador de época (seria a mais forte
in-sample e é puro vazamento do confundidor temporal); nível bruto dos outros reatores como
feature (a margem já carrega a parte relativa sem trazer o efeito de planta); interações
explícitas margem × nível (árvores capturam sozinhas). E as que não dá para construir com o que
existe no repositório: variáveis de processo (temperatura, pressão, concentração de soda),
histórico de condenação de batelada e medição de espessura do revestimento — essas três são o
que realmente falta para a etapa supervisionada ter chance.

## A nova régua: regra composta

Juntando o que a etapa não supervisionada encontrou, a regra a bater deixa de ser o patamar fixo:

**`max diário > 20 ppm`  OU  (`mediana diária > 3.0` E `margem cross-reator > 0.6`)**

Os dois ramos existem porque o sinal aparece de duas formas e nenhuma regra única pega as duas:

- **impulso** — uma leitura altíssima isolada (C3 23/11/2013, 264 ppm) não move a mediana diária;
  só o ramo do `max` pega;
- **elevação sustentada** — vários dias em patamar acima do normal, que o `max` de um dia não
  distingue de um pico de laboratório; o ramo da mediana pega, e a **margem** é o que separa
  "este reator subiu" de "a planta subiu".

**Ressalva obrigatória:** os limiares foram escolhidos olhando as mesmas 31 falhas. Isto é **ponto
de partida para a etapa supervisionada validar no split temporal**, não resultado validado.

In [ ]:
df_detectores = comparar_detectores(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
                                   margem=margem_cross_reator)
df_detectores

### O que segue para a etapa supervisionada

| Achado | Como entra na Etapa 3 |
|---|---|
| Margem cross-reator (lift 5.3–6.9x) | features `margem_max_15d`, `margem_media_15d` |
| Limite móvel de 365 d (imune ao drift) | feature `razao_limite_movel` |
| Idade de campanha (pico em 1-2 anos) | features `idade_campanha`, `faixa_campanha` |
| Qualidade de janela (8 a 150 amostras) | features `n_amostras_15d`, `n_amostras_3d`, `maior_lacuna_15d` |
| Regra composta (F2 0.268 contra 0.132 da vigente) | linha de base a superar, no lugar dos 5 ppm |
| Clusterização de janelas | **descartada** — não separa nada além do limiar |

# Classificação (Abordagem Supervisionada)

In [ ]:
JANELAS       = [15, 12, 9, 6, 3]
N_SPLITS      = 5
RANDOM_STATE  = 42
TEST_SIZE     = 0.2

# Sobrescreve o STATS_FUNCS padrão do utils.py para as células desta seção
STATS_FUNCS = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

# Pista A (ver "Política de tratamento de dados"): treino e avaliação SEMPRE sobre o
# Original (bruto + limpar_excursoes). Os tratamentos deletam as leituras-gatilho e
# invalidam a comparação com a regra vigente (no IQR nada passa de 4.0 ppm).
MAP_MEDICOES = {
    "Original": {"C1": df_crystallizer1, "C2": df_crystallizer2, "C3": df_crystallizer3},
}

CONFIGS = {
    "C1": {"Original": (df_crystallizer1, df_eventos_crystallizer1)},
    "C2": {"Original": (df_crystallizer2, df_eventos_crystallizer2)},
    "C3": {"Original": (df_crystallizer3, df_eventos_crystallizer3)},
}

# As features de contexto (margem cross-reator, limite móvel, EWMA/CUSUM, idade de campanha,
# suporte da janela) entram por CONTEXTO, montado na seção não supervisionada. Rodando com
# `contexto=None` o conjunto volta a ser o antigo — é assim que o ganho fica auditável.
#
# ATENÇÃO: os relatórios abaixo usam o split temporal único do `dividir_dados`, que corta
# pelo quantil dos POSITIVOS e INVERTE a prevalência entre treino e teste (no unificado:
# 80% de positivos no treino contra 25% no teste). Um modelo treinado assim prevê positivo
# quase sempre — daí o recall 1.00 com precisão igual à taxa base. Eles ficam aqui como
# histórico; a conclusão sobre modelo está na seção "Validação", com CV temporal.

## Escopo do ajuste: unificado, e por quê

**Aqui existiam três blocos de classificação — um por reator — que foram removidos.** A decisão
de escopo do estudo é: **ajuste unificado, alarme e prestação de contas por reator** (a tabela
completa está em "Resultados por reator", no fim do notebook). O motivo de o ajuste ser unificado:

| Reator | Falhas | Janelas pré-falha antes do 1º corte temporal |
|---|---|---|
| C1 | 11 | 40 |
| C2 | 14 | 45 |
| C3 | **6** | **15** |
| unificado | 31 | **100** |

Um modelo ajustado só no C3 — justamente o reator onde o sinal existe — teria 15 janelas
positivas para aprender. A célula abaixo mede isso fora da amostra em vez de argumentar.

In [ ]:
# Treinar unificado, só no próprio reator, ou nos outros dois?
# Métrica: PR-AUC dividida pela taxa base do próprio reator (as taxas base diferem entre eles).
df_escopo_treino = comparar_escopo_treino(df_janelas)

### Conclusão do escopo

Nenhum escopo vence nos três reatores, e o resultado que fecha a questão é o do **C3**: treinado
**nos outros dois** ele chega a lift 4.83, contra 3.05 unificado e apenas **1.56 treinado em si
mesmo**. Com 6 falhas, o próprio reator não tem material para ensinar nada.

Some-se que a melhor feature do estudo — a **margem cross-reator** — só existe com os três séries
juntas: separar completamente não é sequer possível.

**Portanto: o ajuste (features e modelos) roda unificado.** O que passa a ser por reator é o
**limiar de alarme** e o **relatório**, nas seções seguintes.

## Ajuste unificado (canônico)

Este é o único ajuste supervisionado do estudo. Ele usa as janelas de cada evento recortadas na
série do **próprio** reator (`map_medicoes_unif`) e as features de contexto (`CONTEXTO`) — ou seja,
unifica a amostra sem misturar a medição de reatores diferentes dentro de uma janela.

In [ ]:
MODO = "Unificado"

# Pista A: só o Original entra nos modelos (ver "Política de tratamento de dados")
CONFIGS_UNIFICADO = {
    "Original": df_crystallizer123,
}

# map_medicoes_unif corrige um defeito que existia aqui: sem ele, a janela de cada evento era
# recortada sobre o dataframe concatenado dos três reatores (182 amostras em vez das 65 do
# reator do evento), então max/p90/range vinham de outro equipamento.
resultados_por_tratamento_unif, df_cv_consolidado_unif = rodar_classificacao_por_tratamento(
    MODO, CONFIGS_UNIFICADO, df_eventos_crystallizer123, JANELAS,
    stats_funcs=STATS_FUNCS, test_size=TEST_SIZE, usar_smote=False,
    map_medicoes_unif=MAP_MEDICOES["Original"], contexto=CONTEXTO
)

# Validação: protocolo, fora da amostra e custo

Esta seção existe porque o relatório supervisionado acima **não sustenta conclusão**. O corte
do `dividir_dados` usa o quantil dos positivos e a prevalência inverte entre treino e teste
(unificado: 24 positivos / 6 negativos no treino, 7 / 21 no teste). Um modelo treinado com 80%
de positivos prevê positivo quase sempre — e é o que se vê: **recall 1.00 em 11 das 12
combinações**, com precisão igual à taxa base do bloco de teste. Um dos blocos do C3 chega a
F1 = 1.00 com cinco linhas.

O que esta seção faz, em ordem:

1. **CV temporal de janela expansiva**, com a prevalência de cada fold visível e F2 (a métrica
   alvo) no lugar de F1 — nas duas tabelas de evento, LC 10 e LC 5;
2. **modelo avaliado como detector**, que é a única comparação justa com a regra vigente:
   pontuar janelas, agrupar alarmes em 15 dias, contar VP/FP contra as falhas do mesmo período;
3. **validação da regra composta fora da amostra** — holdout temporal, leave-one-reactor-out e
   bootstrap;
4. **lead time** evento a evento, o número que decide o projeto;
5. **ponto de operação por custo**, para quando a planta trouxer o custo do alarme falso e o
   da falha perdida.

## 1. CV temporal nas duas tabelas de evento

Treina no passado, testa no bloco seguinte, repete em quatro cortes. Modelos fixos e
balanceados por classe, sem grid interno: com ~30 eventos por fold, um GridSearchCV dentro do
fold escolhe hiperparâmetro por ruído.

In [ ]:
# LC 10 ppm — negativos "difíceis" (excursão clara que não terminou em falha)
feat_lc10, cols_lc10 = extrair_features(
    "Unificado", df_crystallizer123, df_eventos_crystallizer123, JANELAS, STATS_FUNCS,
    map_medicoes_unif=MAP_MEDICOES["Original"], contexto=CONTEXTO)

print("=" * 70); print("LC 10 ppm"); print("=" * 70)
df_cv_lc10, resumo_cv_lc10 = avaliar_cv_temporal(feat_lc10, cols_lc10)

In [ ]:
# LC 5 ppm — o limiar em que a planta realmente opera, com ~3x mais negativos
feat_lc5, cols_lc5 = extrair_features(
    "Unificado", df_crystallizer123, df_eventos_crystallizer123_lc5, JANELAS, STATS_FUNCS,
    map_medicoes_unif=MAP_MEDICOES["Original"], contexto=CONTEXTO)

print("=" * 70); print("LC 5 ppm (operacional)"); print("=" * 70)
df_cv_lc5, resumo_cv_lc5 = avaliar_cv_temporal(feat_lc5, cols_lc5)

print("\n" + "=" * 70)
print("Comparação LC 10 vs LC 5 (F2 médio entre folds)")
print(pd.concat([resumo_cv_lc10["F2_medio"].rename("LC 10"),
                 resumo_cv_lc5["F2_medio"].rename("LC 5")], axis=1).to_string())

### Leitura da CV temporal

O padrão que aparece é o mesmo nos dois LCs: **o primeiro fold tem F2 quase perfeito e os
seguintes desabam para perto de zero**. Não é instabilidade de modelo — é a prevalência: o
bloco de teste do fold 1 tem 89% de positivos, os blocos seguintes têm 12% e 22%. O modelo
aprendeu a prevalência do treino, não o fenômeno.

Com desvio entre folds maior que a diferença entre modelos (F2_dp ≈ 0.5 contra diferenças de
0.1), **os três modelos são indistinguíveis nesta amostra**. Qualquer ranking de modelo neste
conjunto é ruído.

## 2. Modelo avaliado como detector

Aqui o modelo é treinado sobre as ~5000 janelas deslizantes (rótulo `pre_falha`, taxa base 3%,
sem classe negativa sintetizada por limiar), pontua o bloco seguinte, e as janelas acima do
quantil viram alarme agrupado em 15 dias. A regra composta e a regra vigente são medidas **no
mesmo período de teste** — sem isso a comparação não vale.

In [ ]:
res_detector = treinar_detector_janelas(
    df_janelas, MAP_EVENTOS_FALHA,
    margem=margem_cross_reator, map_medicoes=MAP_MEDICOES_BASE
)

### O resultado que decide o rumo da Etapa 3

Fora da amostra, no mesmo período de teste (2 folds, 1491 janelas pontuadas, 10 falhas):

| Regra | VP | FP | precisão | recall | F2 |
|---|---|---|---|---|---|
| **regra composta (20 / 3.5 / 0.9)** | 3 | 11 | 0.214 | 0.300 | **0.278** |
| regra composta (20 / 3.0 / 0.6) | 3 | 18 | 0.143 | 0.300 | 0.246 |
| modelo (quantil 0.90) | 2 | 33 | 0.057 | 0.200 | 0.133 |
| regra vigente (`max > 5`) | 2 | 42 | 0.045 | 0.200 | 0.119 |
| modelo (quantis 0.95 / 0.98 / 0.99) | 0 | 5 / 0 / 0 | 0.000 | 0.000 | 0.000 |

**A regra composta bate o modelo supervisionado e mais que dobra o F2 da regra vigente.** O
modelo só empata com a regra vigente no quantil mais permissivo (0.90), e some nos quantis mais
seletivos — ele não consegue colocar as janelas de pré-falha no topo do ranking.

Não é um resultado contra machine learning em geral: é o resultado esperado quando existem 31
positivos, 90% deles concentrados em um período, e a única variável medida é uma série
univariada. O modelo tem graus de liberdade demais para o que a base sustenta.

**Consequência para a Etapa 3: o entregável é a regra, não o modelo.** Três condições sobre
estatísticas diárias são implementáveis em qualquer linguagem da planta e auditáveis pela
operação — e é o único candidato cujo desempenho fora da amostra foi medido.

## 3. A regra composta fora da amostra

Os limiares foram escolhidos olhando as mesmas 31 falhas. Três validações independentes:
holdout temporal (ajusta no passado, mede no futuro), leave-one-reactor-out (ajusta em dois
reatores, mede no terceiro) e bootstrap das falhas (intervalo do F2).

In [ ]:
validacao_regra = validar_regra_composta(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator,
    data_corte="2019-01-01", limiares_fixos=(20, 3.0, 0.6), n_bootstrap=500
)

### Leitura das três validações — a mais importante desta seção

**Holdout temporal: não conclui, e o motivo é o dado.** Depois de 01/01/2019 existem **3
falhas**. Nenhuma regra acerta nenhuma delas — nem a composta, nem a vigente (que gera 86
alarmes falsos no mesmo período). Com 3 positivos não há como validar nem invalidar: o teste
não tem poder. Isto **não** é evidência de que a regra não funciona; é evidência de que **esta
base não permite validação temporal**, o que precisa constar de qualquer entrega.

**Leave-one-reactor-out: o sinal é do C3.**

| Reator avaliado | VP | FP | recall | F2 |
|---|---|---|---|---|
| C3 | 5 de 6 | 29 | **0.833** | **0.431** |
| C1 | 2 de 11 | 11 | 0.182 | 0.175 |
| C2 | 1 de 14 | 16 | 0.071 | 0.068 |

Com limiares ajustados **nos outros dois reatores**, a regra encontra 5 das 6 falhas do C3 e
quase nada no C1 e no C2. Isso costura com tudo que já sabíamos: as duas paradas de emergência
que a planilha atribui ao ferro são do C3, e o C3 é o reator de outro trem (correlação 0.11
com os demais). **A hipótese "ferro prevê falha" só se sustenta no C3.**

**Bootstrap:** F2 = 0.264, IC95% **[0.150, 0.381]**. O limite inferior fica acima do F2 da
regra vigente (0.132), mas o intervalo é largo — como esperado com 31 positivos.

## 4. Lead time — o número que decide o projeto

In [ ]:
df_lead = lead_time_regra(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator,
    limiares=(20, 3.0, 0.6), dias_max=60
)

df_lead[df_lead["alarmou"]].sort_values("lead_dias", ascending=False)

### Existe antecedência — para metade das falhas alarmadas

A regra composta alarma em **15 das 31 falhas** dentro de 60 dias, com lead mediano de
**3.0 dias** (p25 0.5, p75 21.5, máximo 40). Onze falhas têm lead ≥ 1 dia e **sete têm lead ≥
7 dias**.

É melhor que o diagnóstico anterior (lead mediano 0.0 nos limiares puros de 7/10/20 ppm), e
muda a conversa com a planta: para cerca de um quarto das falhas há **uma semana ou mais** de
antecedência; para as demais, o alarme chega junto com o evento. O produto honesto é
**"antecipação parcial + confirmação com menos alarme falso"**, não "predição de falha".

## 5. Ponto de operação por custo

Enquanto o histórico de condenação indevida e o custo de parada não planejada não chegam da
planta, o ponto de operação é escolhido pelo F2 — que embute uma razão arbitrária (recall vale
4x a precisão). Com os dois custos, a escolha vira aritmética. A tabela abaixo mostra qual
regra vence para cada razão custo(falha perdida)/custo(alarme falso).

In [ ]:
print(sensibilidade_custo(df_detectores, n_falhas=31).to_string(index=False))

print("\nCusto esperado com razão 25:1 (uma falha perdida vale 25 alarmes falsos):")
custo_esperado(df_detectores, custo_fp=1, custo_fn=25, n_falhas=31).head(6)

### Fecho da validação

| Pergunta | Resposta medida |
|---|---|
| Os modelos supervisionados superam a regra? | **Não.** Fora da amostra, F2 0.104 contra 0.278 da regra composta |
| A regra composta supera a regra vigente? | **Sim, dentro da amostra** (0.268 vs 0.132) e no período de teste do detector (0.278 vs 0.119); IC95% do bootstrap [0.150, 0.381] |
| A regra se valida no futuro? | **Indeterminado** — só 3 falhas depois de 2019 |
| A regra vale para os três reatores? | **Não** — recall 0.83 no C3, 0.18 no C1, 0.07 no C2 |
| Existe antecedência? | **Parcial** — 15 de 31 alarmadas, lead mediano 3 dias, 7 falhas com ≥ 7 dias |
| Qual ponto de operação? | Depende da razão de custo: até ~5:1 vence `max > 20`; de 10:1 em diante vencem as regras compostas |

**Recomendação de entrega:** a regra composta como sistema de alarme, com escopo declarado
(desempenho comprovado no C3, indeterminado no C1/C2), acompanhada da ficha de eventos para a
planta validar as datas e do pedido formal dos dados de custo. Modelo supervisionado só volta à
mesa com variáveis de processo além do ferro ou com mais eventos rotulados.

# IsolationForest sobre janelas deslizantes

## O que foi removido, e por quê

A versão anterior treinava o IsolationForest sobre as **59 linhas da tabela de eventos**
(`JANELAS = [6, 3, 1]`, 4 estatísticas, grid de 108 combinações). Problemas:

1. 59 linhas não sustentam um grid de 108 combinações — o "melhor F2 de treino" que escolhia o
   modelo é ruído de seleção, e a matriz de confusão saía de 6 a 12 linhas de teste;
2. detectar anomalia **sobre a tabela de eventos é circular**: a tabela foi construída selecionando
   ultrapassagens, então "anômalo" e "evento" eram quase sinônimos por construção;
3. não havia comparação com taxa base — não dava para saber se o alarme significava alguma coisa.

Agora ele roda onde faz sentido: sobre **toda a operação** (as janelas deslizantes), sem rótulo no
ajuste, avaliado por **lift** e pela mesma convenção de alarme dos demais detectores.

In [ ]:
res_iforest = rodar_iforest_janelas(
    df_janelas, df_janelas_falha, map_falhas=MAP_EVENTOS_FALHA,
    contaminacao_grid=(0.01, 0.02, 0.05, 0.10)
)

In [ ]:
fig_iforest = plotar_iforest_janelas(res_iforest, contaminacao=0.05)
fig_iforest.show()

### Conclusão do IsolationForest

O lift é real (3x a 9x conforme a contaminação), mas o **F2 fica em 0.03–0.08**, uma ordem de
grandeza abaixo da regra composta (0.268) e da própria regra vigente (0.132): ele marca 5% do tempo
para capturar 8 das 31 falhas, gerando 79 alarmes falsos.

Diagnóstico: o IsolationForest mede **distância da operação normal em todas as direções**, e a
maior parte dessa distância é composta por coisas que nada têm a ver com falha — lacuna de
amostragem, mudança de regime da série, janela com poucas amostras. Ou seja, ele reencontra o mesmo
teto da clusterização.

**Decisão: não seguir por detecção de anomalia genérica.** O caminho que carrega sinal é o
direcional — margem cross-reator e nível relativo ao próprio baseline —, já incorporado às features
da etapa supervisionada.

# Cartas de controle

Avaliando EWMA e CUSUM: as cartas disparam com antecedência em relação à falha?

Duas mudanças em relação à versão anterior desta seção:

1. o EWMA passa a usar **baseline rolante** em vez do baseline escolhido à mão em 2012-2015 —
   a série não é estacionária (tau de Kendall -0.29), então um baseline fixo envelhece;
2. as duas cartas deixaram de ser só gráfico: `ewma_z` e `cusum_rel` no instante da âncora
   viraram **features** dos modelos (ver "Insumos" na seção não supervisionada). O CUSUM é uma
   das poucas features que sobrevivem ao ajuste por época (AUC ajustada 0.567).

## EWMA

In [ ]:
# EWMA com BASELINE ROLANTE — substitui o baseline escolhido à mão em 2012-2015.
#
# Por quê: a série tem tendência de queda confirmada (tau de Kendall -0.29, p astronômico;
# inclinação de Sen ~ -0.05 ppm/ano; mediana 2.45 -> 1.85 ppm entre 2011-2015 e 2021-2026).
# Um baseline fixo naquele período deixa o limite sistematicamente frouxo no regime atual, e o
# grid antigo compensava isso levando L até 13 — ou seja, ajustando ruído. No baseline rolante a
# média e a dispersão vêm de uma janela móvel do passado (shift(1) impede que o ponto de hoje
# entre no próprio limite) e a dispersão é robusta (IQR/1.349), porque a série tem excursões de
# três ordens de grandeza.
#
# Leitura esperada da comparação abaixo: com o mesmo L, o baseline rolante dispara MAIS que o
# fixo — a dispersão do regime atual é bem menor que a do baseline de 2012-2015, então o limite
# acompanha a série em vez de ficar alto e imóvel. Esse é o comportamento correto para uma série
# não estacionária, mas significa que a carta precisa de L maior para servir como alarme isolado.
# O valor principal do EWMA rolante aqui não é a carta: é a feature "ewma_z" (e a "cusum_rel"),
# que entram nos modelos — o CUSUM é uma das poucas que sobrevivem ao ajuste por época
# (AUC ajustada 0.567, contra 0.524 da mediana).
serie_fe = (df_crystallizer1.set_index("TIMESTAMP")["Resultado de Ferro (ppm)"]
            .dropna().sort_index())
df_eventos_c1 = df_eventos_crystallizer1.copy()

# --- versão antiga: baseline fixo escolhido à mão
periodos_baseline = [("2012-03-01", "2013-03-25"), ("2014-01-01", "2015-01-01")]
serie_baseline = pd.concat([serie_fe.loc[i:f] for i, f in periodos_baseline])
media_historica, std_historico = serie_baseline.mean(), serie_baseline.std()
print(f"Baseline fixo | média {media_historica:.3f} ppm | std {std_historico:.3f} ppm "
      f"| {len(serie_baseline)} amostras")

df_ewma_fixo = calcular_ewma(serie_fe, media_historica, std_historico, lambd=0.2, L=7)
_, _, f2_fixo = avaliar_carta_controle(df_ewma_fixo, df_eventos_c1,
                                       "EWMA baseline fixo (2012-2015), L=7", janela_dias=15)

# --- versão nova: baseline rolante
df_ewma = calcular_ewma_rolante(serie_fe, lambd=0.2, L=7, janela_baseline=500)
_, _, f2_rolante = avaliar_carta_controle(df_ewma, df_eventos_c1,
                                          "EWMA baseline rolante, L=7", janela_dias=15)

print(f"\nAlarmes: baseline fixo {int(df_ewma_fixo['Alarme'].sum())} "
      f"| baseline rolante {int(df_ewma['Alarme'].sum())}")
print(f"F2: fixo {f2_fixo:.4f} | rolante {f2_rolante:.4f}")

fig_ewma = plotar_carta_ewma(df_ewma, df_eventos_c1,
                             titulo="Carta EWMA com baseline rolante — Crystallizer #1")
fig_ewma.show()

## CUMSUM

In [ ]:
# ==========================================
# 2. FUNÇÃO DE PLOTAGEM
# ==========================================

# ==========================================
# 3. BLOCO DE EXECUÇÃO E OTIMIZAÇÃO
# ==========================================

# Preparação dos dados
df_c1 = df_crystallizer1.copy()
df_c1['TIMESTAMP'] = pd.to_datetime(df_c1['TIMESTAMP'])
df_c1 = df_c1.sort_values('TIMESTAMP').set_index('TIMESTAMP')
serie_fe = df_c1['Resultado de Ferro (ppm)'].dropna()

df_eventos_c1 = df_eventos_crystallizer1.copy()
df_eventos_c1['TIMESTAMP'] = pd.to_datetime(df_eventos_c1['TIMESTAMP'])

# Definição do Grid de Parâmetros para buscar a melhor performance
param_grid_cusum = {
    'janela_baseline': [15, 30, 45, 60], # Quantos dias passados compõem o "normal"
    'k': [0.25, 0.5, 0.75],              # Folga (menor = soma desvios menores)
    'h': [3, 4, 5, 6],                   # Limite de alarme (maior = mais rigoroso)
    'janela_dias': [3, 6, 9, 12]         # Janela de antecedência para prever a falha
}

# Roda a otimização
melhores_parametros, max_f2 = otimizar_cusum_dinamico(serie_fe, df_eventos_c1, param_grid_cusum)

# Gera a carta final com os hiperparâmetros campeões
df_cusum_otimizado = calcular_cusum_dinamico(
    serie_fe, 
    janela_baseline=melhores_parametros['janela_baseline'], 
    k=melhores_parametros['k'], 
    h=melhores_parametros['h']
)

# Avalia formalmente para exibir o relatório
nome_modelo = f"CUSUM Dinâmico (jan_base={melhores_parametros['janela_baseline']}, k={melhores_parametros['k']}, h={melhores_parametros['h']})"
avaliar_carta_controle(df_cusum_otimizado, df_eventos_c1, nome_carta=nome_modelo, janela_dias=melhores_parametros['janela_dias'])

# Plota o gráfico para análise visual (a função devolve a fig; quem chama dá o .show())
fig_cusum = plotar_carta_cusum(
    df_cusum_otimizado, df_eventos_c1,
    titulo=f"Carta CUSUM Dinâmico - C1 {nome_modelo}",
    mostrar_falsos=False
)
fig_cusum.show()

# Resultados por reator

Até aqui todo resultado foi reportado somando os três reatores. Esta seção separa — e a
separação muda a leitura do projeto.

**A regra de trabalho do estudo, medida e não arbitrada:**

| Etapa | Decisão | Evidência |
|---|---|---|
| Carga, limpeza e EDA distribucional | **unificar** | distribuições estatisticamente idênticas: Kruskal-Wallis η² = 0.0008; mediana 2.2 / 2.3 / 2.3 ppm; p99 = 4.3 nos três |
| Construção de features | **unificar (obrigatório)** | margem e posto cross-reator só existem com as três séries |
| Ajuste (modelo e limiar) | **unificar** | o C3 tem 6 falhas e aprende melhor com os outros dois (lift 4.83) do que consigo (1.56) |
| Calibração do alarme | **por reator** | ver a tabela de calibração abaixo |
| Avaliação e relatório | **por reator, sempre** | o número agregado esconde que a regra funciona no C3 e falha no C2 |

## A qual estatística cada reator responde

Antes de calibrar, vale ver *o que* dispara em cada equipamento. Os três têm a mesma distribuição
de ferro, mas não a mesma relação entre ferro e falha.

In [ ]:
linhas_lift = []
for _cryst in MAP_MEDICOES_BASE:
    _J = df_janelas[df_janelas["Crystallizer"] == _cryst]
    _JF = janelas_nas_falhas({_cryst: MAP_MEDICOES_BASE[_cryst]},
                             {_cryst: MAP_EVENTOS_FALHA[_cryst]}, contexto=CONTEXTO)
    _abs = avaliar_lift_series({_cryst: diario_mediana[_cryst]}, _J, _JF, [3.0, 3.5],
                               rotulo=f"{_cryst} mediana diária")
    _mar = avaliar_lift_series({_cryst: MAP_MARGEM[_cryst]}, _J, _JF, [0.6, 1.2],
                               rotulo=f"{_cryst} margem")
    linhas_lift.append(pd.concat([_abs, _mar], ignore_index=True))

df_lift_por_reator = pd.concat(linhas_lift, ignore_index=True)
df_lift_por_reator

### Cada reator tem um gatilho diferente

| Critério (lift) | C1 | C2 | C3 |
|---|---|---|---|
| mediana diária > 3.5 | 1.42 (2 de 11) | 2.58 (4 de 14) | **6.17 (5 de 6)** |
| margem > 0.6 | **4.71 (4 de 11)** | 1.65 (2 de 14) | 5.73 (5 de 6) |
| margem > 1.2 | **12.24 (2 de 11)** | 2.58 (1 de 14) | 6.64 (2 de 6) |

**C3 responde a nível, C1 responde a margem, C2 quase não responde.** É por isso que um limiar
único é subótimo: ele precisa servir a três comportamentos diferentes ao mesmo tempo.

## Calibração do alarme por reator

A **estrutura** da regra continua a mesma nos três (`max > A` OU (`mediana > B` E `margem > C`)) —
o que muda é o ponto de corte. Isso é importante para a implantação: é uma regra só, com uma
tabela de três linhas de parâmetro, e não três lógicas diferentes para a operação decorar.

**Ressalva obrigatória:** com 6 a 14 falhas por reator, o limiar próprio é ajustado *in-sample* e
é um teto otimista. Deve ser lido como "quanto se ganharia se o limiar fosse do equipamento", e
revisto quando a planta trouxer mais eventos rotulados.

In [ ]:
LIMIARES_POR_REATOR, df_calibracao, LIMIAR_GLOBAL = calibrar_limiares_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator
)

In [ ]:
df_regra_calibrada, ALARMES_CALIBRADOS = avaliar_regra_calibrada(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, LIMIARES_POR_REATOR,
    margem=margem_cross_reator, limiares_globais=(20, 3.0, 0.6)
)

### O ganho da calibração

| Regra | VP | FP | precisão | recall | F2 |
|---|---|---|---|---|---|
| **composta com limiar por reator** | **13** | **60** | 0.178 | **0.419** | **0.330** |
| composta com limiar único (20/3.0/0.6) | 11 | 70 | 0.136 | 0.355 | 0.268 |
| regra vigente (`max > 5 ppm`) | 8 | 171 | 0.045 | 0.258 | 0.132 |

Calibrar por reator entrega **duas falhas a mais com dez alarmes falsos a menos** que o limiar
único. Contra a regra vigente: **+5 falhas detectadas (13 contra 8) com 65% menos alarme falso**
(60 contra 171).

Por reator, o ganho é muito desigual — e é essa desigualdade que a próxima tabela expõe:
no C1 o limiar global já era o ótimo (nada muda), no C2 o F2 sobe de 0.118 para 0.183, e no C3 o
mesmo recall de 0.833 passa a custar **16 alarmes falsos em vez de 29**.

## Quadro consolidado por reator

Uma linha por equipamento, com tudo que a equipe precisa ver junto. **Não há linha de total de
propósito:** a média entre reatores esconde exatamente o fato que decide a implantação.

In [ ]:
df_resumo_reator = resumo_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    df_janelas=df_janelas, margem=margem_cross_reator,
    limiares_por_reator=LIMIARES_POR_REATOR, df_lead=df_lead, contexto=CONTEXTO
)
# df_resumo_reator.to_csv(DIR_OUTPUT + "resumo_por_reator.csv", sep=";", index=False)
df_resumo_reator

### Leitura reator a reator

**Crystallizer #3 — onde o método funciona.** 6 falhas, 5 detectadas (recall 0.833) com 16 alarmes
falsos, F2 **0.556** (a regra vigente entrega 0.132 no conjunto). É o reator das duas paradas de
emergência que a própria planilha de inspeção atribui ao ferro (264 ppm em 23/11/2013 e 999 ppm em
02/08/2018, ambas com furo confirmado na abertura), é o de outro trem (correlação 0.11 com os
demais) e é o de campanha mais longa (mediana de 1516 dias na falha). **Contrapartida: o lead time
mediano é de 1 dia** — no C3 o ferro confirma, não antecipa.

**Crystallizer #1 — sinal na margem, com antecedência.** 5 de 11 falhas, F2 0.357. Responde à
margem cross-reator (lift 12.24 acima de 1.2), não ao nível. E é o reator com **melhor lead time:
mediana de 15 dias** nas 4 falhas alarmadas — é onde existe janela real de decisão.

**Crystallizer #2 — o ferro não prevê, e provavelmente não é problema de método.** 14 falhas, 3
detectadas, F2 0.183 mesmo com limiar próprio. Lendo os apontamentos: *"quebra dos parafusos da pá
superior do Hidro#1"*, *"Dano na Hélice"*, *"vazamento pela região do selo/mesa"*, além de vários
vazamentos localizados em bocais. São falhas de **agitador e de bocal**, não de revestimento do
casco — e ferro no licor não tem por que subir nesses modos. O C2 tem **0 menções a ferro** e
**1 troca de reator** em 14 paradas, contra 2 menções e 1 troca em apenas 6 paradas no C3.

**Consequência:** parte do que estamos exigindo que o ferro preveja não é previsível por ferro,
**por construção**. Isso não se resolve com algoritmo — se resolve pedindo à planta a
**classificação do modo de falha** de cada evento (a ficha de validação já vai para eles; basta
acrescentar essa coluna). Filtrando para falhas de revestimento, o denominador do recall cai e o
desempenho medido sobe sem nenhuma mudança de método.

---

# Resumo executivo — leitura para a equipe Bayer

*Documento de fechamento desta etapa. Todos os números abaixo são reproduzíveis pelas células
deste notebook e foram medidos sobre os mesmos dados: as três séries de ferro do laboratório
(2011-2026) e a planilha de inspeção dos vitrificados.*

## 1. A pergunta

O ferro medido no licor consegue **antecipar a falha do vitrificado** dos reatores de PIA, com
menos alarme falso do que a regra em uso (ferro acima de 5 ppm)?

O estudo anterior da equipe Bayer respondeu que não: dos 10 eventos analisados, **1** tinha ferro
acima de 5 ppm antes; e entre 2017 e 2025 houve **256 ultrapassagens, 247 sem falha** — cerca de
**3,5% de precisão**. Este trabalho parte daí, refaz a base de eventos e mede tudo contra taxa
base.

## 2. O que mudou na montagem do problema

Três correções que alteram qualquer conclusão posterior:

1. **A base de eventos foi refeita a partir da planilha de inspeção**, não do deck. Dos 100
   apontamentos, 60 caem no período da série; 8 foram descartados por coincidirem com **parada de
   planta** (lacuna simultânea nos três reatores, que não é falha de equipamento); apontamentos do
   mesmo reator a menos de 7 dias foram fundidos. Resultado: **31 falhas** (C1 11, C2 14, C3 6).
2. **Os eventos foram reancorados para o fim da amostragem**, porque quando o reator para o
   laboratório também para: 39 dos 60 apontamentos caíam dentro de uma lacuna, e sem reancorar a
   "janela anterior à falha" continha a própria parada.
3. **Só o valor fisicamente impossível é descartado** (uma amostra de 20000 ppm em toda a base).
   As leituras altas isoladas ficam: são ultrapassagens que não terminaram em falha, ou seja, o
   falso positivo que qualquer detector precisa enfrentar. O corte antigo de 100 ppm apagava
   justamente as duas leituras que a planilha registra como **causa** de parada de emergência
   (264 ppm em 23/11/2013 e 999 ppm em 02/08/2018, ambas no C3, ambas com furo confirmado).

## 3. Principais achados

**1) A regra de 5 ppm não funciona — mas não porque o ferro seja inútil.** Ela dispara em 25,8%
das janelas que antecedem falha e em 23,2% de uma janela qualquer: **lift 1,11**, praticamente
indistinguível do acaso. O limiar está no lugar errado, não a variável: `max > 20 ppm` tem lift
6,24 e a mediana diária ≥ 3,5 ppm tem lift 2,18.

**2) O que informa não é o nível de ferro — é a comparação entre os reatores.** A **margem**
(mediana diária do reator menos a mediana dos três) tem lift **5,3x a 6,9x**, contra 3,7x do
melhor critério de nível absoluto, com a mesma taxa de alarme. Como são três séries, a mediana é o
valor do meio: a margem é positiva só para o reator que está liderando, e mede o quanto ele lidera.

**3) A série caiu ao longo dos anos e isso contamina toda análise ingênua.** Tendência de queda
confirmada (tau de Kendall −0,29, p astronômico nos três; mediana diária de 2,45-2,50 ppm em
2011-2015 para 1,80-1,90 em 2021-2026). Como **90% das falhas ocorreram até 2018**, ferro e falha
caem juntos sem que um cause o outro — e qualquer variável correlacionada com o tempo parece
preditiva sem prever nada. Corrigindo por época, a "melhor feature" (mediana da janela) cai de AUC
0,678 para **0,524**, ou seja, era um relógio.

**4) Uma regra composta supera a regra vigente com folga.** `max diário > 20 ppm` **OU**
(`mediana diária > 3,0` **E** `margem > 0,6`) — os dois ramos são necessários porque o sinal
aparece de duas formas: o impulso (uma leitura altíssima isolada, que não move a mediana) e a
elevação sustentada (que o máximo de um dia não distingue de um pico de laboratório).

**5) Calibrar o limiar por reator melhora ainda mais** — mesma estrutura de regra, três linhas de
parâmetro: **13 falhas detectadas contra 8 da regra vigente, com 60 alarmes falsos contra 171**.

**6) Modelos de machine learning não superaram a regra.** Medidos fora da amostra e no mesmo
período de teste, F2 0,133 do melhor modelo contra 0,278 da regra composta e 0,119 da regra
vigente. Com 31 eventos, 90% deles concentrados em um período e uma única variável medida, o
modelo tem mais liberdade do que a base sustenta. **Clusterização e detecção de anomalia genérica
(IsolationForest) foram testadas e descartadas** — a primeira reencontra apenas o limiar, a
segunda gera 79 alarmes falsos para capturar 8 falhas.

**7) O desempenho é muito diferente entre reatores, e isso é o achado mais acionável.**

In [ ]:
# Tabela de fechamento — regenerada a partir dos objetos do notebook
print("=" * 78)
print("DESEMPENHO GLOBAL (31 falhas, alarmes agrupados em 15 dias)".center(78))
print("=" * 78)
print(df_regra_calibrada.to_string(index=False))

print()
print("=" * 78)
print("DESEMPENHO POR REATOR (limiar calibrado)".center(78))
print("=" * 78)
print(df_resumo_reator[["reator", "falhas", "responde_a", "limiar_calibrado",
                        "VP", "FP", "precisao", "recall", "F2",
                        "falhas_com_alarme_60d", "lead_mediano_dias"]].to_string(index=False))

### O que a tabela por reator diz

| Reator | Situação | Recomendação |
|---|---|---|
| **C3** | 5 de 6 falhas detectadas (recall 0,83), F2 **0,556**, 16 alarmes falsos. É o reator das duas paradas que a planilha atribui ao ferro. **Lead time mediano de 1 dia.** | **Implantar.** O ferro confirma a falha com muito menos alarme falso — mas não antecipa. |
| **C1** | 5 de 11 falhas, F2 0,357. Responde à **margem**, não ao nível. **Lead time mediano de 15 dias** nas falhas alarmadas. | **Implantar em modo de acompanhamento.** É onde existe janela real de decisão. |
| **C2** | 3 de 14 falhas, F2 0,183 mesmo com limiar próprio. | **Não implantar ainda** — investigar antes o modo de falha (ver abaixo). |

**A hipótese mais provável para o C2 não é estatística.** Lendo os apontamentos: *"quebra dos
parafusos da pá superior do Hidro#1"*, *"Dano na Hélice"*, *"vazamento pela região do selo/mesa"*,
além de vários vazamentos localizados em bocais. São falhas de **agitador e de bocal**, não de
revestimento do casco — e ferro no licor não tem por que subir nesses modos. O C2 tem **0 menções
a ferro** e **1 troca de reator** em 14 paradas; o C3 tem 2 menções e 1 troca em apenas 6.

## 4. Limitações que precisam ser declaradas junto com o resultado

Estas ressalvas fazem parte da entrega. Nenhuma delas invalida os achados, mas todas mudam o que
se pode prometer:

1. **Os limiares foram escolhidos olhando as mesmas 31 falhas.** O bootstrap dá F2 = 0,264 com
   intervalo de 95% entre **0,150 e 0,381** — o piso ainda fica acima da regra vigente (0,132),
   mas o intervalo é largo.
2. **Não é possível validar no futuro com esta base.** Depois de 01/01/2019 existem **3 falhas**,
   e nenhuma regra acerta nenhuma delas — inclusive a vigente, que gera 86 alarmes falsos no mesmo
   período. Isso não é evidência de que a regra falha; é evidência de que **o teste não tem poder**.
3. **A antecipação é parcial.** No conjunto, 15 das 31 falhas recebem alarme em até 60 dias, com
   lead mediano de 3 dias e 7 falhas com uma semana ou mais. O produto honesto é
   **"antecipação parcial + confirmação com muito menos alarme falso"**, não "predição de falha".
4. **As datas dos eventos ainda precisam de confirmação.** O deck da equipe e a planilha de
   inspeção divergem em pelo menos três eventos. A ficha de validação gerada neste notebook
   (60 apontamentos, com data original, data reancorada, deslocamento e motivo) existe para ser
   revisada linha a linha pela planta.
5. **Não há variável de processo nem histórico de condenação.** Sem custo de alarme falso e custo
   de parada não planejada, o ponto de operação é escolhido por F2 — que embute uma razão
   arbitrária. A função de custo já está montada e parametrizada: **basta a planta informar a razão
   de custo para a escolha virar aritmética.**

## 5. O que pedimos à planta para a próxima etapa

Em ordem de impacto sobre o resultado:

1. **Classificação do modo de falha de cada evento** (revestimento/casco, bocal, agitador, outros)
   — provavelmente resolve o C2 e corrige o denominador do recall dos três;
2. **Confirmação das datas** dos 60 apontamentos da ficha de validação;
3. **Custo de um alarme falso** (condenação indevida) e **custo de uma parada não planejada** —
   define o ponto de operação;
4. **Variáveis de processo** do circuito (temperatura, pressão, concentração de soda, vazão) —
   é o único caminho para o aprendizado de máquina voltar a fazer sentido: hoje a base é
   univariada;
5. **Eventos rotulados posteriores a 2019**, se existirem em outra fonte — sem eles a validação
   temporal continuará sem poder.

## 6. O que está pronto para entrega

**A regra composta, com limiar por reator** — não um modelo. Três condições sobre estatísticas
diárias do próprio laboratório, implementáveis em qualquer linguagem da planta e auditáveis pela
operação:

```
ALARME  se   máximo diário de ferro > A ppm
        ou  ( mediana diária > B ppm  E  margem sobre a mediana dos três reatores > C ppm )
```

| Reator | A | B | C | Escopo recomendado |
|---|---|---|---|---|
| C1 | 30 | 2,5 | 0,6 | acompanhamento (lead mediano 15 dias) |
| C2 | 20 | 3,0 | 0,6 | não implantar antes de classificar o modo de falha |
| C3 | 30 | 3,5 | 0,9 | implantar (recall 0,83; confirma a falha, não antecipa) |

Onde **margem = mediana diária do reator − mediana diária dos três reatores no mesmo dia** —
a única grandeza deste estudo que exige os três reatores medidos no mesmo dia, e a que mais
carrega informação.

Junto com a regra vão: a **ficha de validação de eventos**, a **tabela de sensibilidade a custo**
(qual regra vence para cada razão custo-de-falha / custo-de-alarme) e este notebook, com cada
decisão documentada na célula em que foi tomada.